# 5. Evolutionary LGCA

Identity-based LGCA assign a persistent label to every cell. Cell
properties can then be inherited and changed at birth, linking
spatial population dynamics to trait evolution.

This lesson develops the evolutionary go-and-grow scenario from
the legacy evolutionary notebook using an explicit current
`ModelSpec` and several stochastic replicates.

**Learning objectives**

- construct an identity-based model and seed its spatial state;
- add heritable birth-rate variation to the pipeline;
- compare replicate population trajectories; and
- distinguish one simulation result from a scientific conclusion.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from lgca.model import (
    AnalysisSpec,
    Description,
    ModelSpec,
    SpaceSpec,
    StateSpec,
    TimeSpec,
    run_model,
)
from lgca.pipeline import InteractionPipelineSpec
from lgca.simulation import DensityRecorder, NodeRecorder, PopulationRecorder


## Biological question and model

We ask how heritable variation in proliferation changes expansion
from a populated boundary. The model is deliberately small enough
for exploration. The `ib.birthdeath` interaction initializes a
birth-rate property `r_b`; daughter cells inherit a mutated value
with spread controlled by `std`.

The complete specification remains visible. The Boolean initial
array indicates occupied channels; the identity-based backend
replaces those occupancies with persistent labels.


In [ ]:
def initial_edge_population(length=40, channels=4):
    nodes = np.zeros((length, channels), dtype=bool)
    nodes[0, :] = True
    return nodes


def make_evolution_spec(seed, steps=30):
    return ModelSpec(
        description=Description(
            title="Evolutionary boundary expansion",
            details="Identity-based birth and death with heritable birth-rate variation.",
        ),
        space=SpaceSpec(
            geometry="lin",
            dims=(40,),
            boundary="reflecting",
        ),
        state=StateSpec(
            nodes=initial_edge_population(),
            restchannels=2,
            identity_based=True,
        ),
        time=TimeSpec(
            steps=steps,
            seed=seed,
        ),
        dynamics=InteractionPipelineSpec(
            operators=[
                {
                    "name": "ib.birthdeath",
                    "parameters": {
                        "r_b": 0.2,
                        "r_d": 0.01,
                        "std": 0.05,
                        "a_max": 1.0,
                        "track_inheritance": False,
                    },
                }
            ],
        ),
        analysis=AnalysisSpec(
            observers=[NodeRecorder(), DensityRecorder(), PopulationRecorder()],
        ),
    )


demonstration_spec = make_evolution_spec(seed=51)
demonstration_spec


## Run stochastic replicates

A seed identifies a realization; it is not a biological replicate.
Nevertheless, independent seeds are the computational replicates
needed to estimate stochastic variability under fixed parameters.


In [ ]:
seeds = (51, 52, 53)
replicate_results = {
    seed: run_model(make_evolution_spec(seed), showprogress=False)
    for seed in seeds
}

fig, axis = plt.subplots(figsize=(7, 3.8), constrained_layout=True)
for seed, replicate in replicate_results.items():
    axis.plot(replicate.lgca.n_steps, replicate.lgca.n_t, label=f"seed {seed}")
axis.set(xlabel="time step", ylabel="population", title="Evolutionary replicate trajectories")
axis.legend()
plt.show()
plt.close(fig)


Differences between trajectories arise from stochastic deaths,
births, mutations and movement. Reporting only the most interesting
trajectory would bias the analysis. A study should prespecify the
number of seeds and summarize their distribution.


## Inspect the inherited trait among living cells

Identity labels stored in the lattice index the property arrays.
We collect the birth-rate value for every living cell in the final
state. Repeated labels would represent repeated references to the
same cell, but volume exclusion stores each living label once.


In [ ]:
final_traits = {}
for seed, replicate in replicate_results.items():
    labels = np.asarray(replicate.lgca.nodes[replicate.lgca.nonborder]).ravel()
    labels = labels[labels > 0].astype(int)
    final_traits[seed] = np.asarray(replicate.lgca.props["r_b"])[labels]

fig, axis = plt.subplots(figsize=(7, 3.8), constrained_layout=True)
bins = np.linspace(0.0, 0.5, 16)
for seed, values in final_traits.items():
    axis.hist(values, bins=bins, alpha=0.45, label=f"seed {seed}")
axis.set(xlabel="inherited birth rate r_b", ylabel="living cells")
axis.legend()
plt.show()
plt.close(fig)


## What would support a scientific conclusion?

These three short replicates demonstrate the workflow, not an
evolutionary claim. A scientific conclusion would require:

- a baseline with `std=0` to separate evolution from demography;
- enough prespecified replicate seeds to quantify uncertainty;
- sensitivity analysis for death rate, initial population and domain size;
- an observable tied to the hypothesis, such as front speed or the
  birth-rate distribution at the expanding edge; and
- convergence checks showing that conclusions do not depend on the
  shortened teaching lattice or run length.

## Exercises

1. Compare `std=0` and `std=0.05` using paired seeds.
2. Define the occupied front position and plot it over time.
3. Increase the number of replicates and add a mean trajectory with
   a percentile interval.
4. Separate trait values at the front from those in the bulk.
